# MorphAgent — End‑to‑End Demo

This notebook walks you through a **complete, minimal MorphAgent run** on a tiny
5‑sample dataset, step by step.

What it demonstrates:

1. Inspect a ready‑made demo dataset (5 neuron microscopy samples, with
   precomputed segmentation masks + image slices).
2. Reuse the **existing knowledge** (`expert_knowledge`, `deep_research`, `RAG`)
   — offline, **without needing PaddleX** or a GPU.
3. Run the full MorphAgent pipeline for **2 rounds × 5 features per round**
   (dataset understanding → knowledge injection → feature planning → code +
   VLM feature extraction → validation → `features.csv`).
4. Inspect the outputs.

> **Cost / scale.** This is deliberately tiny (5 samples, 2×5 features) so it
> finishes quickly and cheaply. Segmentation is **skipped** because masks are
> already provided, so **no GPU is required**. You only need a working LLM + VLM
> API (configured in the next step).


## 0. Environment setup (one-time)

MorphAgent runs with **no local model** — everything goes through an
OpenAI-compatible API. There are only two setup steps; both are handled here.

**Step A — conda environment (run once in a terminal).** You cannot build the
kernel's own env from inside it, so create it in a shell, register it as a
Jupyter kernel, then pick it from the kernel selector (top-right of this
notebook):

```bash
cd <repo-root>
conda env create -f envs/environment.yml           # creates the `morphagent` env
conda activate morphagent
python -m ipykernel install --user --name morphagent --display-name "Python (morphagent)"
```

Then select **Python (morphagent)** as this notebook's kernel.

**Step B — API credentials.** Fill them in the next code cell (they are applied
to the whole pipeline, including the subprocess run in step 4). Run the cells
**top to bottom**; if you change the credentials later, re-run from the API cell
below (or restart the kernel).


In [ ]:
import os

# ======================= API CONFIGURATION =======================
# Fill in your OpenAI-compatible endpoints/keys/models below.
# These are read by config.py at import time, so this cell MUST run
# before the environment-check cell. Re-run from here if you change them.
os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"   # text model endpoint
os.environ["LLM_API_KEY"]  = ""                             # <-- your LLM key
os.environ["LLM_MODEL"]    = "gpt-4o"                       # planning / code / review

os.environ["VLM_BASE_URL"] = "https://api.openai.com/v1"   # vision model endpoint
os.environ["VLM_API_KEY"]  = ""                             # <-- your VLM key
os.environ["VLM_MODEL"]    = "gpt-4o"                       # scores image features

# Optional: deep-research model used by --auto-deep-research (falls back to LLM_*)
# os.environ["DEEP_RESEARCH_BASE_URL"] = "https://api.perplexity.ai"
# os.environ["DEEP_RESEARCH_API_KEY"]  = ""
# os.environ["DEEP_RESEARCH_MODEL"]    = "sonar-deep-research"

# Optional: raise PubMed / Europe PMC rate limits for --auto-literature-retrieval
# os.environ["NCBI_EMAIL"] = "you@example.org"

# PaddleX device for PDF parsing (cpu is the default; set gpu:0 if you installed paddlepaddle-gpu)
os.environ.setdefault("PADDLEX_DEVICE", "cpu")
# =================================================================

missing = [k for k in ("LLM_API_KEY", "VLM_API_KEY") if not os.environ.get(k)]
if missing:
    print("[!] Not set yet:", ", ".join(missing),
          "-- fill them above before running the pipeline (step 4).")
else:
    print("API configured:", os.environ["LLM_MODEL"], "(LLM) /",
          os.environ["VLM_MODEL"], "(VLM)")


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# --- Locate the repository root (this notebook lives in <repo>/demo/) ---------
here = Path.cwd()
candidates = [here, here.parent] + list(here.parents)
REPO_ROOT = None
for p in candidates:
    if (p / "main.py").exists() and (p / "demo").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the MorphAgent repo root. Run this notebook from inside "
        "the repository (it should live at <repo>/demo/morphagent_demo.ipynb)."
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEMO_DIR     = REPO_ROOT / "demo"
DEMO_DATA    = DEMO_DIR / "data"          # project root: contains dataset/ + knowledge folders
DATASET_DIR  = DEMO_DATA / "dataset"
DEMO_RESULTS = DEMO_DIR / "results"
PRECOMPUTED  = DEMO_DIR / "precomputed"

print("Repo root   :", REPO_ROOT)
print("Demo data   :", DEMO_DATA)
assert DATASET_DIR.exists(), f"missing dataset dir: {DATASET_DIR}"

# Quick import check (fails early if the environment is incomplete)
import config  # noqa: F401
import langchain_core, langchain_openai, langgraph  # noqa: F401
print("Imports OK — environment looks good.")


## 1. Check the API configuration

This re-reads the values you set in the **API configuration** cell above, prints
the endpoints/models MorphAgent will use, and (optionally) sends one tiny message
to confirm the LLM credentials work. If a key is empty or the ping fails, fix the
values in the API cell above, **re-run it and this cell**.


In [ ]:
import importlib, config
importlib.reload(config)   # re-read env vars set in the API cell above
from config import settings

print("LLM  base_url:", settings.llm_base_url)
print("LLM  model   :", settings.llm_model)
print("VLM  base_url:", settings.vlm_online_base_url)
print("VLM  model   :", settings.vlm_online_model)
print("Sandbox conda env      :", settings.conda_env)
print("Segmentation conda env :", settings.segmentation_conda_env)

if not settings.llm_api_key:
    print("\n[!] LLM_API_KEY is EMPTY. Set LLM_* (and VLM_*) env vars or edit config.py,")
    print("    then restart the kernel before running the pipeline.")
else:
    # Optional connectivity ping (comment out to skip)
    try:
        from langchain_core.messages import HumanMessage
        from config import make_chat_llm
        reply = make_chat_llm().invoke([HumanMessage(content="reply with the single word: ok")])
        print("\nLLM ping reply:", getattr(reply, "content", reply))
    except Exception as e:
        print("\n[!] LLM ping failed:", repr(e))
        print("    Check LLM_BASE_URL / LLM_API_KEY / LLM_MODEL and try again.")


## 2. Inspect the demo dataset

The demo dataset follows MorphAgent's layout: **one dataset = one directory, one
sample per subdirectory**. Each sample carries a primary `image.tif`, a PNG
`slices/` view for the VLM, and a `segmentation/` folder with masks.

```
demo/data/                     <- project root (pass this to --data-root)
├── dataset/
│   ├── dataset_index.txt      <- free-text description read by the LLM
│   ├── WT_1/
│   │   ├── image.tif
│   │   ├── slices/slice_0000.png
│   │   └── segmentation/{mask_cell,mask_nucleus,mask_bundle,mask_filament,mask_droplet}.tif
│   └── WT_2 ... WT_5
├── expert_knowledge/          <- notes + example image
├── deep_research/             <- report.md (text, no PaddleX needed)
└── RAG/                       <- a few example PDFs
```


In [ ]:
from utils_helpers import read_dataset_index, find_description_file

samples = read_dataset_index(DATASET_DIR)
print("Samples found:", samples)

desc_path = find_description_file(DATASET_DIR)
print("Description file:", desc_path)
print("-" * 70)
print(desc_path.read_text(encoding="utf-8")[:1200])

# Show the segmentation masks available for the first sample
s0 = DATASET_DIR / samples[0]
masks = sorted(p.name for p in (s0 / "segmentation").glob("*.tif"))
print("-" * 70)
print(f"Masks in {samples[0]}/segmentation:", masks)
print("(these become seg['mask_cell'], seg['mask_nucleus'], ... in generated code)")


In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt

sid = samples[0]
sdir = DATASET_DIR / sid

img = tifffile.imread(sdir / "image.tif").astype(np.float32)
if img.ndim == 3:                       # (C,H,W) or (H,W,C) -> take a 2D view
    img = img[0] if img.shape[0] <= 4 else img[..., 0]

def norm(a):
    a = a.astype(np.float32)
    lo, hi = np.percentile(a, 1), np.percentile(a, 99)
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

cell = tifffile.imread(sdir / "segmentation" / "mask_cell.tif")
nuc  = tifffile.imread(sdir / "segmentation" / "mask_nucleus.tif")

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(norm(img), cmap="gray");   ax[0].set_title(f"{sid} — image.tif (MIP)")
ax[1].imshow(norm(img), cmap="gray")
ax[1].contour(cell > 0, colors="cyan", linewidths=0.6)
ax[1].contour(nuc  > 0, colors="magenta", linewidths=0.6)
ax[1].set_title("cell (cyan) + nucleus (magenta)")
try:
    from PIL import Image
    ax[2].imshow(Image.open(sdir / "slices" / "slice_0000.png"), cmap="gray")
    ax[2].set_title("slices/slice_0000.png (VLM view)")
except Exception as e:
    ax[2].set_title(f"slice unavailable: {e}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## 3. Reuse the existing knowledge (offline, no PaddleX needed for this demo)

MorphAgent can inject three kinds of external knowledge into feature planning.
This demo keeps the **existing results** and makes them run **without needing
to re-parse PDFs**:

- **`expert_knowledge/`** — read as text/image and summarized by the LLM.
- **`deep_research/`** — shipped as `report.md` (plain text), read directly.
- **`RAG/`** — the cell below **pre-seeds the RAG cache** from
  `demo/precomputed/`, so the pipeline finds a valid cache and skips PDF parsing.

In a real run you can instead generate these automatically:
- `--auto-deep-research` — one API call writes a markdown report into `deep_research/`.
- `--auto-literature-retrieval` — searches PubMed/Europe PMC and downloads
  open-access PDFs into `RAG/`, then parses them with PaddleX (installed in the
  unified environment; CPU by default).

Segmentation is likewise reused: each sample already has masks, so the pipeline
marks them `skipped_user_seg` and never invokes Cellpose-SAM (no GPU needed).


In [ ]:
from datetime import datetime
from knowledge.rag import _compute_rag_folder_hash

def seed_rag_cache(project_root: Path, precomputed_summary: Path) -> Path:
    """Write a valid RAG cache from a precomputed summary so extract_rag_knowledge
    returns it directly (skipping PaddleX PDF parsing). The hash is computed over
    the actual RAG folder on disk, so it always matches at run time."""
    rag_dir = project_root / "RAG"
    pdfs = sorted(rag_dir.glob("*.pdf"))
    xmls = sorted(rag_dir.glob("*.xml"))
    if not pdfs and not xmls:
        raise RuntimeError(f"No PDF/XML files in {rag_dir}")
    rag_hash = _compute_rag_folder_hash(rag_dir, pdfs, xmls)
    cache_dir = project_root / ".rag_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"rag_cache_{rag_hash}.txt"
    content = precomputed_summary.read_text(encoding="utf-8")
    meta = {"hash": rag_hash, "created_at": datetime.now().isoformat(),
            "content_length": len(content)}
    with open(cache_file, "w", encoding="utf-8") as f:
        f.write("# RAG Cache Metadata\n")
        f.write(f"# {json.dumps(meta, ensure_ascii=False)}\n")
        f.write("# End Metadata\n\n")
        f.write(content)
    return cache_file

cache_file = seed_rag_cache(DEMO_DATA, PRECOMPUTED / "rag_knowledge_summary.txt")
print("RAG cache seeded:", cache_file)
print("deep_research report:", (DEMO_DATA / "deep_research" / "report.md").exists())
print("expert_knowledge files:", [p.name for p in (DEMO_DATA / "expert_knowledge").iterdir()])


## 4. Run MorphAgent (2 rounds × 5 features)

We launch the full pipeline via `main.py` as a subprocess (the most robust way —
`main.py` uses multiprocessing internally). Key flags:

| Flag | Value | Meaning |
|------|-------|---------|
| positional `user_query` | task text | drives feature planning |
| `--data-root` | `demo/data` | project root (auto‑detects `dataset/`) |
| `--num-rounds` | `2` | two feature‑extraction rounds ("epochs") |
| `--features-per-iteration` | `5` | plan 5 features per round |
| `--target-feature-count` | `10` | stop around 2×5 features |
| `--method` | `both` | use code **and** VLM features |
| `--segmentation-skip-if-present` | (default) | reuse provided masks; no GPU |

The run streams its log below. It will call your LLM/VLM API, so it takes a few
minutes depending on the endpoint. Outputs go to `demo/results/demo_run/`.


In [ ]:
RUN_DIR = DEMO_RESULTS / "demo_run"

cmd = [
    sys.executable, "main.py",
    "Generate unbiased morphological features that quantify Tau protein "
    "aggregation and neuronal structure in these images",
    "--data-root", str(DEMO_DATA),
    "--results-dir", str(RUN_DIR),
    "--num-rounds", "2",
    "--features-per-iteration", "5",
    "--target-feature-count", "10",
    "--method", "both",
    "--segmentation-skip-if-present",
]

# Make the sandbox / segmentation subprocess use the SAME conda env as this
# kernel, regardless of its name (falls back to the config defaults otherwise).
env = os.environ.copy()
cur_env = os.environ.get("CONDA_DEFAULT_ENV")
if cur_env and cur_env != "base":
    env.setdefault("CONDA_ENV", cur_env)
    env.setdefault("SEGMENTATION_CONDA_ENV", cur_env)

print("Running:\n  " + " ".join(repr(c) if " " in c else c for c in cmd) + "\n")

proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\n=== main.py exited with code {proc.returncode} ===")


## 5. Inspect the results

Load the extracted feature table and the planning/validation artifacts written to
`demo/results/demo_run/`.


In [ ]:
import pandas as pd

print("Result files in", RUN_DIR, ":")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(RUN_DIR))

feat_csv = RUN_DIR / "features.csv"
if feat_csv.exists():
    df = pd.read_csv(feat_csv)
    print("\nfeatures.csv shape:", df.shape)
    print("feature columns:", [c for c in df.columns if c != "sample_id"])
    display(df)
else:
    print("\n[!] features.csv not found — check the run log above for errors.")

# Show the per-round feature plans (the 5 features proposed each round)
for rnd in sorted(RUN_DIR.glob("round_*/feature_plan.json")):
    plan = json.loads(rnd.read_text(encoding="utf-8"))
    feats = plan.get("features", plan) if isinstance(plan, dict) else plan
    print(f"\n=== {rnd.parent.name} feature plan ===")
    if isinstance(feats, list):
        for f in feats:
            name = (f.get("name") or f.get("feature_name")) if isinstance(f, dict) else f
            desc = f.get("description", "") if isinstance(f, dict) else ""
            print(f"  - {name}: {desc[:90]}")


## Done — where to go next

You just ran the full MorphAgent loop on a tiny dataset. To adapt it to your own
data:

1. **Bring your own dataset** — mirror the layout under `demo/data/`: one
   `dataset/<sample_id>/image.tif` per sample, an optional `slices/` PNG view,
   and (optionally) precomputed masks under `segmentation/`. Add a free‑text
   `dataset_index.txt` describing the biology and channels. See the top‑level
   `README.md` → *Input data format* for the full spec.
2. **Let MorphAgent segment for you** — drop the `segmentation/` folders and run
   without `--segmentation-skip-if-present` to auto‑segment with Cellpose‑SAM
   (needs a GPU), or use the Allen backend in `segmentation_allen/`.
3. **Add real knowledge** — put your own PDFs/notes in `expert_knowledge/`,
   `deep_research/`, `RAG/`. Raw PDFs require the optional PaddleX extra
   (`pip install -r envs/requirements-optional.txt`); `.md`/`.txt`/`.xml`
   sources work without it.
4. **Scale up** — increase `--num-rounds`, `--features-per-iteration`, and
   `--target-feature-count` for a full feature bank.

See `installation_skill.md` and `README.md` for the complete reference.
